In [1]:
from google.colab import drive
drive.mount('/content/drive')
!ls -1h /content/drive/My\Drive/Collaboration_XAI/XAI_IS

Mounted at /content/drive
101_EyeTracking_PreProcessed.dat
all_cs_data_binary.csv
all_cs_data_binary_reduced.csv
All_CS_rifat_data.csv
All_CS_rifat_data_reduced_feature_multiclass.csv
assets
attack_data
Attack_detection.ipynb
AUC-ROC_curve.png
Average_mitigation_comp_30_new1.pdf
Average_mitigation_comp_30_new.pdf
BIM_0.1_10.npy
BIM_attack_sign_0.004_10_csv.csv
BIM_attack_sign_0.004_10.npy
BIM_LSTM__attack_signature.eps
boston_model.pt
checkpoint
CL_report_DT.PNG
CL_report_EBM.PNG
CL_report_LR.PNG
CS_dataset_PRV_UNL.pdf
CS_rifat_4_class.csv
CS_rifat_reduced_4_class.csv
cybersickness_Explanation_model.pt
cybersickness_model.pt
cybersickness_transformer_outputs
data_class_renamed_CSS.csv
data_class_renamed.csv
data_explorartion.py
dataset.csv
dataset_reduced.csv
df_battery_calce.csv
df_battery_nasa.csv
DP_SGD_activity_transformer2.pdf
DP_SGD_activity_transformer2update.pdf
DP_SGD_DL_Ehtask_accuracy_fixed_edit_new_ep.pdf
DP_SGD_DL_gaze_prediction.pdf
DP_SGD_DL_sensor_accuracy_new.pdf
DP_SG

In [2]:
import math
import os
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, Dataset

In [3]:
class Config:
    # This will be updated automatically after upload in Colab.
    csv_path: str = "/content/drive/My Drive/Collaboration_XAI/XAI_IS/All_CS_rifat_data.csv"
    target_col: str = "fms"

    # Set to a real identity/session column if your CSV has one.
    # Examples: subject_id, participant_id, user_id, session_id
    # Leave as None if unavailable.
    private_label_col: Optional[str] = None

    # Windowing
    seq_len: int = 64
    stride: int = 16

    # Training
    batch_size: int = 32
    num_workers: int = 2

    # Transformer
    d_model: int = 128
    nhead: int = 8
    num_layers: int = 4
    dim_feedforward: int = 256
    dropout: float = 0.2

    lr_stage1: float = 1e-3
    lr_stage2_main: float = 1e-4
    lr_stage2_adv: float = 1e-4
    weight_decay: float = 1e-4

    epochs_stage1_utility: int = 12
    epochs_stage1_privacy: int = 12
    epochs_stage2: int = 20

    # Privacy-utility tradeoff
    privacy_budget_k: float = 0.50
    lambda_cam: float = 1.0
    lambda_budget: float = 5.0
    gamma_privacy: float = 0.25
    beta_private_suppression: float = 1.0

    grad_clip: float = 1.0
    val_size: float = 0.15
    test_size: float = 0.15
    seed: int = 42
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    save_dir: str = "/content/drive/My Drive/Collaboration_XAI/XAI_IS/cybersickness_transformer_outputs"


CFG = Config()
print(CFG)

In [4]:
# ============================================================
# Cell 2: Reproducibility + Label Mapping + Data Helpers
# ============================================================
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def map_fms_to_4_classes(x: float) -> str:
    """
    Resolved mapping:
        0            -> none
        0 < x <= 2   -> low
        2 < x <= 3   -> medium
        x > 3        -> high
    """
    if pd.isna(x):
        return np.nan
    if x == 0:
        return "none"
    if x <= 2:
        return "low"
    if x <= 3:
        return "medium"
    return "high"


def infer_private_column(df: pd.DataFrame) -> Optional[str]:
    candidates = [
        "subject_id",
        "subject",
        "participant_id",
        "participant",
        "user_id",
        "user",
        "session_id",
        "session",
        "identity",
        "id",
    ]
    for c in candidates:
        if c in df.columns:
            return c
    return None


class SequenceDataset(Dataset):
    def __init__(self, x: np.ndarray, y: np.ndarray, p: Optional[np.ndarray] = None):
        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.p = None if p is None else torch.tensor(p, dtype=torch.long)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx: int):
        if self.p is None:
            return self.x[idx], self.y[idx], torch.tensor(-1, dtype=torch.long)
        return self.x[idx], self.y[idx], self.p[idx]


def build_windows(
    df: pd.DataFrame,
    feature_cols: List[str],
    utility_col: str,
    seq_len: int,
    stride: int,
    private_col: Optional[str] = None,
) -> Tuple[np.ndarray, np.ndarray, Optional[np.ndarray]]:
    x_all = df[feature_cols].to_numpy(dtype=np.float32)
    y_all = df[utility_col].to_numpy()
    p_all = df[private_col].to_numpy() if private_col is not None else None

    xs, ys, ps = [], [], []
    for start in range(0, len(df) - seq_len + 1, stride):
        end = start + seq_len
        xs.append(x_all[start:end])
        ys.append(y_all[end - 1])
        if p_all is not None:
            ps.append(p_all[end - 1])

    x_windows = np.stack(xs)
    y_windows = np.array(ys)
    p_windows = np.array(ps) if p_all is not None else None
    return x_windows, y_windows, p_windows

In [5]:
# ============================================================
# Cell 3: Transformer Encoder + Classifier + Mask Generator
# ============================================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1)]


class TransformerBackbone(nn.Module):
    def __init__(
        self,
        input_dim: int,
        d_model: int,
        nhead: int,
        num_layers: int,
        dim_feedforward: int,
        dropout: float,
        max_len: int,
    ):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len=max_len)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = self.input_proj(x)
        z = self.pos_enc(z)
        h = self.encoder(z)
        h = self.norm(h)
        return h


class TransformerClassifier(nn.Module):
    def __init__(
        self,
        input_dim: int,
        num_classes: int,
        d_model: int,
        nhead: int,
        num_layers: int,
        dim_feedforward: int,
        dropout: float,
        max_len: int,
    ):
        super().__init__()
        self.backbone = TransformerBackbone(
            input_dim=input_dim,
            d_model=d_model,
            nhead=nhead,
            num_layers=num_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            max_len=max_len,
        )
        self.head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, num_classes),
        )

    def forward(self, x: torch.Tensor, return_hidden: bool = False):
        h = self.backbone(x)
        pooled = h.mean(dim=1)
        logits = self.head(pooled)
        if return_hidden:
            return logits, h
        return logits


class TransformerMaskGenerator(nn.Module):
    def __init__(
        self,
        input_dim: int,
        d_model: int,
        nhead: int,
        num_layers: int,
        dim_feedforward: int,
        dropout: float,
        max_len: int,
    ):
        super().__init__()
        self.backbone = TransformerBackbone(
            input_dim=input_dim,
            d_model=d_model,
            nhead=nhead,
            num_layers=max(2, num_layers // 2),
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            max_len=max_len,
        )
        self.mask_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.backbone(x)
        mask = torch.sigmoid(self.mask_head(h)).squeeze(-1)
        return mask

In [6]:
# ============================================================
# Cell 4: Grad-CAM + Losses + Training/Evaluation Helpers
# ============================================================
def compute_token_gradcam(
    model: TransformerClassifier,
    x: torch.Tensor,
    target_labels: torch.Tensor,
) -> torch.Tensor:
    was_training = model.training
    model.eval()

    logits, hidden = model(x, return_hidden=True)
    hidden.retain_grad()
    selected = logits.gather(1, target_labels.view(-1, 1)).sum()

    model.zero_grad(set_to_none=True)
    selected.backward(retain_graph=False)

    grads = hidden.grad
    alpha = grads.mean(dim=-1, keepdim=True)
    cam = torch.relu((alpha * hidden).sum(dim=-1))
    cam = cam / (cam.sum(dim=1, keepdim=True) + 1e-8)

    if was_training:
        model.train()
    return cam.detach()


def get_class_weights(y: np.ndarray, num_classes: int) -> torch.Tensor:
    classes = np.arange(num_classes)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    return torch.tensor(weights, dtype=torch.float32)


def budget_loss(mask: torch.Tensor, k: float) -> torch.Tensor:
    return torch.relu(mask.mean(dim=1) - k).mean()


def cam_alignment_loss(
    mask: torch.Tensor,
    cam_utility: torch.Tensor,
    cam_privacy: Optional[torch.Tensor] = None,
    beta: float = 1.0,
) -> torch.Tensor:
    utility_term = -(mask * cam_utility).sum(dim=1).mean()
    if cam_privacy is None:
        return utility_term
    privacy_term = (mask * cam_privacy).sum(dim=1).mean()
    return utility_term + beta * privacy_term


def train_one_epoch_classifier(
    model: TransformerClassifier,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    device: str,
    use_private_label: bool = False,
    grad_clip: float = 1.0,
) -> Dict[str, float]:
    model.train()
    losses, y_true, y_pred = [], [], []

    for x, y, p in loader:
        x = x.to(device)
        target = p.to(device) if use_private_label else y.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        losses.append(loss.item())
        y_true.extend(target.detach().cpu().numpy().tolist())
        y_pred.extend(logits.argmax(dim=1).detach().cpu().numpy().tolist())

    return {
        "loss": float(np.mean(losses)),
        "acc": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }


@torch.no_grad()
def evaluate_classifier(
    model: TransformerClassifier,
    loader: DataLoader,
    criterion: nn.Module,
    device: str,
    use_private_label: bool = False,
) -> Dict[str, float]:
    model.eval()
    losses, y_true, y_pred = [], [], []

    for x, y, p in loader:
        x = x.to(device)
        target = p.to(device) if use_private_label else y.to(device)
        logits = model(x)
        loss = criterion(logits, target)

        losses.append(loss.item())
        y_true.extend(target.detach().cpu().numpy().tolist())
        y_pred.extend(logits.argmax(dim=1).detach().cpu().numpy().tolist())

    return {
        "loss": float(np.mean(losses)),
        "acc": accuracy_score(y_true, y_pred),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "y_true": y_true,
        "y_pred": y_pred,
    }


@torch.no_grad()
def evaluate_stage2(
    mask_generator: TransformerMaskGenerator,
    utility_model: TransformerClassifier,
    loader: DataLoader,
    device: str,
    adversary: Optional[TransformerClassifier] = None,
) -> Dict[str, float]:
    mask_generator.eval()
    utility_model.eval()
    if adversary is not None:
        adversary.eval()

    util_true, util_pred = [], []
    priv_true, priv_pred = [], []
    keep_ratios = []

    for x, y, p in loader:
        x = x.to(device)
        y = y.to(device)
        p = p.to(device)

        mask = mask_generator(x)
        x_masked = x * mask.unsqueeze(-1)
        logits_u = utility_model(x_masked)

        keep_ratios.extend(mask.mean(dim=1).detach().cpu().numpy().tolist())
        util_true.extend(y.detach().cpu().numpy().tolist())
        util_pred.extend(logits_u.argmax(dim=1).detach().cpu().numpy().tolist())

        if adversary is not None and torch.all(p >= 0):
            logits_p = adversary(x_masked)
            priv_true.extend(p.detach().cpu().numpy().tolist())
            priv_pred.extend(logits_p.argmax(dim=1).detach().cpu().numpy().tolist())

    out = {
        "utility_acc": accuracy_score(util_true, util_pred),
        "utility_f1_macro": f1_score(util_true, util_pred, average="macro", zero_division=0),
        "avg_keep_ratio": float(np.mean(keep_ratios)),
        "utility_y_true": util_true,
        "utility_y_pred": util_pred,
    }

    if adversary is not None and len(priv_true) > 0:
        out["privacy_acc"] = accuracy_score(priv_true, priv_pred)
        out["privacy_f1_macro"] = f1_score(priv_true, priv_pred, average="macro", zero_division=0)
        out["privacy_y_true"] = priv_true
        out["privacy_y_pred"] = priv_pred
    else:
        out["privacy_acc"] = float("nan")
        out["privacy_f1_macro"] = float("nan")

    return out


def train_stage2(
    mask_generator: TransformerMaskGenerator,
    utility_model: TransformerClassifier,
    adversary: Optional[TransformerClassifier],
    baseline_utility: TransformerClassifier,
    train_loader: DataLoader,
    val_loader: DataLoader,
    utility_criterion: nn.Module,
    privacy_criterion: Optional[nn.Module],
    cfg: Config,
    privacy_enabled: bool,
) -> Dict[str, nn.Module]:
    device = cfg.device
    mask_generator.to(device)
    utility_model.to(device)
    baseline_utility.to(device)
    if adversary is not None:
        adversary.to(device)

    for p in baseline_utility.parameters():
        p.requires_grad = False
    baseline_utility.eval()

    baseline_privacy = None
    if privacy_enabled and adversary is not None:
        baseline_privacy = TransformerClassifier(
            input_dim=utility_model.backbone.input_proj.in_features,
            num_classes=adversary.head[-1].out_features,
            d_model=cfg.d_model,
            nhead=cfg.nhead,
            num_layers=cfg.num_layers,
            dim_feedforward=cfg.dim_feedforward,
            dropout=cfg.dropout,
            max_len=cfg.seq_len,
        ).to(device)
        baseline_privacy.load_state_dict(adversary.state_dict())
        for p in baseline_privacy.parameters():
            p.requires_grad = False
        baseline_privacy.eval()

    main_optimizer = torch.optim.AdamW(
        list(mask_generator.parameters()) + list(utility_model.parameters()),
        lr=cfg.lr_stage2_main,
        weight_decay=cfg.weight_decay,
    )

    adv_optimizer = None
    if privacy_enabled and adversary is not None:
        adv_optimizer = torch.optim.AdamW(
            adversary.parameters(),
            lr=cfg.lr_stage2_adv,
            weight_decay=cfg.weight_decay,
        )

    best_f1 = -1.0
    best_state = None

    for epoch in range(1, cfg.epochs_stage2 + 1):
        mask_generator.train()
        utility_model.train()
        if privacy_enabled and adversary is not None:
            adversary.train()

        total_main_loss = []
        total_adv_loss = []

        for x, y, p in train_loader:
            x = x.to(device)
            y = y.to(device)
            p = p.to(device)

            x_cam_u = x.detach().clone().requires_grad_(True)
            cam_u = compute_token_gradcam(baseline_utility, x_cam_u, y)

            cam_p = None
            if privacy_enabled and baseline_privacy is not None and torch.all(p >= 0):
                x_cam_p = x.detach().clone().requires_grad_(True)
                cam_p = compute_token_gradcam(baseline_privacy, x_cam_p, p)

            # Step A: adversary update
            if privacy_enabled and adversary is not None and adv_optimizer is not None and torch.all(p >= 0):
                for prm in adversary.parameters():
                    prm.requires_grad = True

                adv_optimizer.zero_grad(set_to_none=True)
                with torch.no_grad():
                    mask_detached = mask_generator(x)
                    x_masked_detached = x * mask_detached.unsqueeze(-1)
                logits_p = adversary(x_masked_detached)
                loss_adv = privacy_criterion(logits_p, p)
                loss_adv.backward()
                torch.nn.utils.clip_grad_norm_(adversary.parameters(), cfg.grad_clip)
                adv_optimizer.step()
                total_adv_loss.append(loss_adv.item())

            # Step B: mask + utility update
            if adversary is not None:
                for prm in adversary.parameters():
                    prm.requires_grad = False

            main_optimizer.zero_grad(set_to_none=True)
            mask = mask_generator(x)
            x_masked = x * mask.unsqueeze(-1)

            logits_u = utility_model(x_masked)
            loss_u = utility_criterion(logits_u, y)
            loss_b = budget_loss(mask, cfg.privacy_budget_k)
            loss_cam = cam_alignment_loss(mask, cam_u, cam_p, beta=cfg.beta_private_suppression)

            if privacy_enabled and adversary is not None and torch.all(p >= 0):
                logits_priv = adversary(x_masked)
                loss_priv = privacy_criterion(logits_priv, p)
            else:
                loss_priv = torch.tensor(0.0, device=device)

            total_loss = (
                loss_u
                + cfg.lambda_cam * loss_cam
                + cfg.lambda_budget * loss_b
                - cfg.gamma_privacy * loss_priv
            )
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(
                list(mask_generator.parameters()) + list(utility_model.parameters()),
                cfg.grad_clip,
            )
            main_optimizer.step()
            total_main_loss.append(total_loss.item())

        val_metrics = evaluate_stage2(
            mask_generator,
            utility_model,
            val_loader,
            device,
            adversary=adversary if privacy_enabled else None,
        )

        priv_txt = ""
        if privacy_enabled:
            priv_txt = f", val_priv_acc={val_metrics['privacy_acc']:.4f}"

        print(
            f"[Stage2][Epoch {epoch:02d}] "
            f"main_loss={np.mean(total_main_loss):.4f}, "
            f"adv_loss={np.mean(total_adv_loss) if total_adv_loss else 0.0:.4f}, "
            f"val_util_acc={val_metrics['utility_acc']:.4f}, "
            f"val_util_f1={val_metrics['utility_f1_macro']:.4f}, "
            f"keep_ratio={val_metrics['avg_keep_ratio']:.4f}"
            f"{priv_txt}"
        )

        if val_metrics["utility_f1_macro"] > best_f1:
            best_f1 = val_metrics["utility_f1_macro"]
            best_state = {
                "mask_generator": {k: v.cpu() for k, v in mask_generator.state_dict().items()},
                "utility_model": {k: v.cpu() for k, v in utility_model.state_dict().items()},
                "adversary": None if adversary is None else {k: v.cpu() for k, v in adversary.state_dict().items()},
            }

    if best_state is not None:
        mask_generator.load_state_dict(best_state["mask_generator"])
        utility_model.load_state_dict(best_state["utility_model"])
        if privacy_enabled and adversary is not None and best_state["adversary"] is not None:
            adversary.load_state_dict(best_state["adversary"])

    return {
        "mask_generator": mask_generator,
        "utility_model": utility_model,
        "adversary": adversary,
    }

In [8]:
# ============================================================
# Cell 5: Main Training Pipeline
# ============================================================
def main(cfg: Config) -> None:
    set_seed(cfg.seed)
    save_dir = Path(cfg.save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    print(f"Using device: {cfg.device}")

    if not os.path.exists(cfg.csv_path):
        if IN_COLAB:
            print("CSV not found at the configured path. Please upload your CSV now.")
            uploaded = files.upload()
            if len(uploaded) == 0:
                raise FileNotFoundError("No CSV uploaded.")
            uploaded_name = list(uploaded.keys())[0]
            cfg.csv_path = f"/content/{uploaded_name}"
            print(f"Uploaded file detected: {cfg.csv_path}")
        else:
            raise FileNotFoundError(f"CSV not found: {cfg.csv_path}")

    print(f"Reading CSV from: {cfg.csv_path}")
    df = pd.read_csv(cfg.csv_path)
    print(f"CSV shape: {df.shape}")
    print("Columns:", list(df.columns))

    if cfg.target_col not in df.columns:
        raise ValueError(f"Target column '{cfg.target_col}' not found in CSV.")

    # Convert FMS to 4 classes
    df["cybersickness_class"] = df[cfg.target_col].apply(map_fms_to_4_classes)
    df = df.dropna(subset=["cybersickness_class"]).reset_index(drop=True)

    print("Mapped class counts:")
    print(df["cybersickness_class"].value_counts(dropna=False))

    # Resolve privacy label column
    private_col = cfg.private_label_col
    if private_col is None:
        private_col = infer_private_column(df)

    privacy_enabled = private_col is not None and private_col in df.columns
    if privacy_enabled:
        print(f"Privacy adversary ENABLED using private label column: {private_col}")
    else:
        print(
            "Privacy adversary DISABLED because no subject/user/session identity column was found.\n"
            "To enable privacy adversarial training, set CFG.private_label_col to your real identity column."
        )

    excluded = {cfg.target_col, "cybersickness_class"}
    if privacy_enabled:
        excluded.add(private_col)

    numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and c not in excluded]
    if len(numeric_cols) == 0:
        raise ValueError("No numeric feature columns found after excluding target/private columns.")

    df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)
    df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

    # Utility labels
    utility_encoder = LabelEncoder()
    df["utility_id"] = utility_encoder.fit_transform(df["cybersickness_class"])
    print("Utility classes:", list(utility_encoder.classes_))

    # Private labels
    private_encoder = None
    if privacy_enabled:
        private_encoder = LabelEncoder()
        df["private_id"] = private_encoder.fit_transform(df[private_col].astype(str))
        num_private_classes = len(private_encoder.classes_)
        print(f"Number of private classes: {num_private_classes}")
    else:
        df["private_id"] = -1
        num_private_classes = 0

    # Build windows
    x_windows, y_windows, p_windows = build_windows(
        df=df,
        feature_cols=numeric_cols,
        utility_col="utility_id",
        seq_len=cfg.seq_len,
        stride=cfg.stride,
        private_col="private_id",
    )

    print(f"Window tensor shape: {x_windows.shape}")
    print("Window-level class distribution:")
    print(pd.Series(y_windows).value_counts().sort_index())

    # Split
    indices = np.arange(len(x_windows))
    train_idx, temp_idx = train_test_split(
        indices,
        test_size=cfg.val_size + cfg.test_size,
        random_state=cfg.seed,
        stratify=y_windows,
    )
    temp_y = y_windows[temp_idx]
    relative_val_size = cfg.val_size / (cfg.val_size + cfg.test_size)
    val_idx, test_idx = train_test_split(
        temp_idx,
        test_size=1.0 - relative_val_size,
        random_state=cfg.seed,
        stratify=temp_y,
    )

    scaler = StandardScaler()
    x_train_2d = x_windows[train_idx].reshape(-1, x_windows.shape[-1])
    scaler.fit(x_train_2d)

    def transform_x(arr: np.ndarray) -> np.ndarray:
        shp = arr.shape
        return scaler.transform(arr.reshape(-1, shp[-1])).reshape(shp).astype(np.float32)

    x_train = transform_x(x_windows[train_idx])
    x_val = transform_x(x_windows[val_idx])
    x_test = transform_x(x_windows[test_idx])

    y_train, y_val, y_test = y_windows[train_idx], y_windows[val_idx], y_windows[test_idx]
    p_train, p_val, p_test = p_windows[train_idx], p_windows[val_idx], p_windows[test_idx]

    train_ds = SequenceDataset(x_train, y_train, p_train)
    val_ds = SequenceDataset(x_val, y_val, p_val)
    test_ds = SequenceDataset(x_test, y_test, p_test)

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)
    test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)

    num_features = x_train.shape[-1]
    num_utility_classes = len(utility_encoder.classes_)

    # ========================================================
    # Stage 1A: Baseline utility transformer
    # ========================================================
    utility_model_stage1 = TransformerClassifier(
        input_dim=num_features,
        num_classes=num_utility_classes,
        d_model=cfg.d_model,
        nhead=cfg.nhead,
        num_layers=cfg.num_layers,
        dim_feedforward=cfg.dim_feedforward,
        dropout=cfg.dropout,
        max_len=cfg.seq_len,
    ).to(cfg.device)

    utility_weights = get_class_weights(y_train, num_utility_classes).to(cfg.device)
    utility_criterion = nn.CrossEntropyLoss(weight=utility_weights)
    utility_optimizer = torch.optim.AdamW(
        utility_model_stage1.parameters(),
        lr=cfg.lr_stage1,
        weight_decay=cfg.weight_decay,
    )

    print("\n========== Stage 1A: Baseline Utility Transformer ==========")
    best_val_f1 = -1.0
    best_utility_state = None

    for epoch in range(1, cfg.epochs_stage1_utility + 1):
        train_metrics = train_one_epoch_classifier(
            utility_model_stage1,
            train_loader,
            utility_optimizer,
            utility_criterion,
            cfg.device,
            False,
            cfg.grad_clip,
        )
        val_metrics = evaluate_classifier(
            utility_model_stage1,
            val_loader,
            utility_criterion,
            cfg.device,
            False,
        )
        print(
            f"[Stage1-Utility][Epoch {epoch:02d}] "
            f"train_loss={train_metrics['loss']:.4f}, train_acc={train_metrics['acc']:.4f}, train_f1={train_metrics['f1_macro']:.4f}, "
            f"val_loss={val_metrics['loss']:.4f}, val_acc={val_metrics['acc']:.4f}, val_f1={val_metrics['f1_macro']:.4f}"
        )
        if val_metrics["f1_macro"] > best_val_f1:
            best_val_f1 = val_metrics["f1_macro"]
            best_utility_state = {k: v.cpu() for k, v in utility_model_stage1.state_dict().items()}

    if best_utility_state is not None:
        utility_model_stage1.load_state_dict(best_utility_state)

    # ========================================================
    # Stage 1B: Baseline privacy transformer (optional)
    # ========================================================
    privacy_model_stage1 = None
    privacy_criterion = None

    if privacy_enabled and len(np.unique(p_train)) > 1:
        privacy_model_stage1 = TransformerClassifier(
            input_dim=num_features,
            num_classes=num_private_classes,
            d_model=cfg.d_model,
            nhead=cfg.nhead,
            num_layers=cfg.num_layers,
            dim_feedforward=cfg.dim_feedforward,
            dropout=cfg.dropout,
            max_len=cfg.seq_len,
        ).to(cfg.device)

        privacy_weights = get_class_weights(p_train, num_private_classes).to(cfg.device)
        privacy_criterion = nn.CrossEntropyLoss(weight=privacy_weights)
        privacy_optimizer = torch.optim.AdamW(
            privacy_model_stage1.parameters(),
            lr=cfg.lr_stage1,
            weight_decay=cfg.weight_decay,
        )

        print("\n========== Stage 1B: Baseline Privacy Transformer ==========")
        best_priv_f1 = -1.0
        best_priv_state = None

        for epoch in range(1, cfg.epochs_stage1_privacy + 1):
            train_metrics = train_one_epoch_classifier(
                privacy_model_stage1,
                train_loader,
                privacy_optimizer,
                privacy_criterion,
                cfg.device,
                True,
                cfg.grad_clip,
            )
            val_metrics = evaluate_classifier(
                privacy_model_stage1,
                val_loader,
                privacy_criterion,
                cfg.device,
                True,
            )
            print(
                f"[Stage1-Privacy][Epoch {epoch:02d}] "
                f"train_loss={train_metrics['loss']:.4f}, train_acc={train_metrics['acc']:.4f}, train_f1={train_metrics['f1_macro']:.4f}, "
                f"val_loss={val_metrics['loss']:.4f}, val_acc={val_metrics['acc']:.4f}, val_f1={val_metrics['f1_macro']:.4f}"
            )
            if val_metrics["f1_macro"] > best_priv_f1:
                best_priv_f1 = val_metrics["f1_macro"]
                best_priv_state = {k: v.cpu() for k, v in privacy_model_stage1.state_dict().items()}

        if best_priv_state is not None:
            privacy_model_stage1.load_state_dict(best_priv_state)
    else:
        print("\n========== Stage 1B skipped ==========")
        print("No valid privacy label column found, so baseline privacy training is skipped.")

    # ========================================================
    # Stage 2: CAM-guided adversarial learning
    # ========================================================
    print("\n========== Stage 2: CAM-Guided Adversarial Training ==========")

    mask_generator = TransformerMaskGenerator(
        input_dim=num_features,
        d_model=cfg.d_model,
        nhead=cfg.nhead,
        num_layers=cfg.num_layers,
        dim_feedforward=cfg.dim_feedforward,
        dropout=cfg.dropout,
        max_len=cfg.seq_len,
    ).to(cfg.device)

    utility_model_stage2 = TransformerClassifier(
        input_dim=num_features,
        num_classes=num_utility_classes,
        d_model=cfg.d_model,
        nhead=cfg.nhead,
        num_layers=cfg.num_layers,
        dim_feedforward=cfg.dim_feedforward,
        dropout=cfg.dropout,
        max_len=cfg.seq_len,
    ).to(cfg.device)
    utility_model_stage2.load_state_dict(utility_model_stage1.state_dict())

    adversary_stage2 = None
    if privacy_model_stage1 is not None:
        adversary_stage2 = TransformerClassifier(
            input_dim=num_features,
            num_classes=num_private_classes,
            d_model=cfg.d_model,
            nhead=cfg.nhead,
            num_layers=cfg.num_layers,
            dim_feedforward=cfg.dim_feedforward,
            dropout=cfg.dropout,
            max_len=cfg.seq_len,
        ).to(cfg.device)
        adversary_stage2.load_state_dict(privacy_model_stage1.state_dict())

    stage2_models = train_stage2(
        mask_generator=mask_generator,
        utility_model=utility_model_stage2,
        adversary=adversary_stage2,
        baseline_utility=utility_model_stage1,
        train_loader=train_loader,
        val_loader=val_loader,
        utility_criterion=utility_criterion,
        privacy_criterion=privacy_criterion,
        cfg=cfg,
        privacy_enabled=privacy_enabled and adversary_stage2 is not None,
    )

    # ========================================================
    # Final evaluation
    # ========================================================
    print("\n========== Final Test Evaluation ==========")
    test_metrics = evaluate_stage2(
        stage2_models["mask_generator"],
        stage2_models["utility_model"],
        test_loader,
        cfg.device,
        adversary=stage2_models["adversary"] if (privacy_enabled and stage2_models["adversary"] is not None) else None,
    )

    print(
        f"Test Utility Accuracy: {test_metrics['utility_acc']:.4f}\n"
        f"Test Utility Macro-F1: {test_metrics['utility_f1_macro']:.4f}\n"
        f"Average Keep Ratio: {test_metrics['avg_keep_ratio']:.4f}"
    )

    if privacy_enabled and stage2_models["adversary"] is not None:
        chance_level = 1.0 / num_private_classes
        print(
            f"Test Privacy Accuracy: {test_metrics['privacy_acc']:.4f}\n"
            f"Test Privacy Macro-F1: {test_metrics['privacy_f1_macro']:.4f}\n"
            f"Chance Level: {chance_level:.4f}"
        )

    print("\nClassification report (utility task):")
    print(
        classification_report(
            test_metrics["utility_y_true"],
            test_metrics["utility_y_pred"],
            target_names=list(utility_encoder.classes_),
            zero_division=0,
        )
    )

    # Save model checkpoint and predictions
    torch.save(
        {
            "mask_generator": stage2_models["mask_generator"].state_dict(),
            "utility_model": stage2_models["utility_model"].state_dict(),
            "adversary": None if stage2_models["adversary"] is None else stage2_models["adversary"].state_dict(),
            "utility_classes": list(utility_encoder.classes_),
            "private_classes": [] if private_encoder is None else list(private_encoder.classes_),
            "numeric_cols": numeric_cols,
            "config": vars(cfg),
        },
        save_dir / "cybersickness_gazeprotector_transformer.pt",
    )

    pd.DataFrame({
        "utility_true": [utility_encoder.classes_[i] for i in test_metrics["utility_y_true"]],
        "utility_pred": [utility_encoder.classes_[i] for i in test_metrics["utility_y_pred"]],
    }).to_csv(save_dir / "test_predictions.csv", index=False)

    print(f"Saved outputs to: {save_dir}")

In [13]:
# ============================================================
# Cell 6: Run
# ============================================================
main(CFG)

Using device: cpu
Reading CSV from: /content/drive/My Drive/Collaboration_XAI/XAI_IS/All_CS_rifat_data.csv
CSV shape: (20105, 43)
Columns: ['Convergence_distance', 'fms', 'Left_Eye_Openness', 'Right_Eye_Openness', 'LeftPupilDiameter', 'RightPupilDiameter', 'LeftPupilPosInSensorX', 'LeftPupilPosInSensorY', 'RightPupilPosInSensorX', 'RightPupilPosInSensorY', 'GazeOriginLclSpc_X', 'GazeOriginLclSpc_Y', 'GazeOriginLclSpc_Z', 'GazeDirectionLclSpc_X', 'GazeDirectionLclSpc_Y', 'GazeDirectionLclSpc_Z', 'GazeOriginWrldSpc_X', 'GazeOriginWrldSpc_Y', 'GazeOriginWrldSpc_Z', 'GazeDirectionWrldSpc_X', 'GazeDirectionWrldSpc_Y', 'GazeDirectionWrldSpc_Z', 'NrmLeftEyeOriginX', 'NrmLeftEyeOriginY', 'NrmLeftEyeOriginZ', 'NrmRightEyeOriginX', 'NrmRightEyeOriginY', 'NrmRightEyeOriginZ', 'NrmSRLeftEyeGazeDirX', 'NrmSRLeftEyeGazeDirY', 'NrmSRLeftEyeGazeDirZ', 'NrmSRRightEyeGazeDirX', 'NrmSRRightEyeGazeDirY', 'NrmSRRightEyeGazeDirZ', 'HeadQRotationX', 'HeadQRotationY', 'HeadQRotationZ', 'HeadQRotationW', 'Head

/tmp/ipykernel_2766/3451884794.py:42: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)



========== Stage 1A: Baseline Utility Transformer ==========
[Stage1-Utility][Epoch 01] train_loss=1.3457, train_acc=0.3968, train_f1=0.3151, val_loss=1.1981, val_acc=0.4894, val_f1=0.4057
[Stage1-Utility][Epoch 02] train_loss=1.1017, train_acc=0.4857, train_f1=0.4142, val_loss=1.2432, val_acc=0.5106, val_f1=0.3810
[Stage1-Utility][Epoch 03] train_loss=1.0041, train_acc=0.5165, train_f1=0.4527, val_loss=0.9727, val_acc=0.5585, val_f1=0.4856
[Stage1-Utility][Epoch 04] train_loss=0.8936, train_acc=0.6043, train_f1=0.5365, val_loss=0.9583, val_acc=0.5053, val_f1=0.4768
[Stage1-Utility][Epoch 05] train_loss=0.8338, train_acc=0.5804, train_f1=0.5411, val_loss=1.0255, val_acc=0.6011, val_f1=0.5143
[Stage1-Utility][Epoch 06] train_loss=0.7487, train_acc=0.6579, train_f1=0.6115, val_loss=1.4328, val_acc=0.6170, val_f1=0.4992
[Stage1-Utility][Epoch 07] train_loss=0.6782, train_acc=0.6796, train_f1=0.6361, val_loss=1.4738, val_acc=0.5957, val_f1=0.5015
[Stage1-Utility][Epoch 08] train_loss=0.76

/tmp/ipykernel_2766/3451884794.py:42: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


[Stage2][Epoch 01] main_loss=0.3028, adv_loss=0.0000, val_util_acc=0.5904, val_util_f1=0.4929, keep_ratio=0.5041
[Stage2][Epoch 02] main_loss=-0.0222, adv_loss=0.0000, val_util_acc=0.6223, val_util_f1=0.5036, keep_ratio=0.4963
[Stage2][Epoch 03] main_loss=-0.0477, adv_loss=0.0000, val_util_acc=0.6277, val_util_f1=0.5202, keep_ratio=0.4963
[Stage2][Epoch 04] main_loss=-0.0788, adv_loss=0.0000, val_util_acc=0.6436, val_util_f1=0.5294, keep_ratio=0.4909
[Stage2][Epoch 05] main_loss=-0.0946, adv_loss=0.0000, val_util_acc=0.6543, val_util_f1=0.5413, keep_ratio=0.5000
[Stage2][Epoch 06] main_loss=-0.1351, adv_loss=0.0000, val_util_acc=0.6755, val_util_f1=0.5433, keep_ratio=0.4962
[Stage2][Epoch 07] main_loss=-0.1326, adv_loss=0.0000, val_util_acc=0.6702, val_util_f1=0.5492, keep_ratio=0.4943
[Stage2][Epoch 08] main_loss=-0.1513, adv_loss=0.0000, val_util_acc=0.6596, val_util_f1=0.5277, keep_ratio=0.4902
[Stage2][Epoch 09] main_loss=-0.1641, adv_loss=0.0000, val_util_acc=0.6596, val_util_f1=0

In [9]:
# ============================================================
# Cell 7: Masking ratio setup
# ============================================================
# You wanted:
# 25%, 25%, 25%
# 50%, 50%, 50%
# 75%, 75%, 75%
#
# That means 3 repeats for each masking ratio.

masking_ratios = [0.25, 0.50, 0.75]   # fraction MASKED
num_repeats = 3

def mask_ratio_to_keep_ratio(mask_ratio: float) -> float:
    # In this framework, privacy_budget_k = fraction KEPT
    return 1.0 - mask_ratio

In [ ]:
# ============================================================
# Cell 8: Run experiments for different masking ratios
# ============================================================
from sklearn.metrics import accuracy_score, f1_score

all_results = []

base_save_root = "/content/cybersickness_masking_experiments"

for mask_ratio in masking_ratios:
    keep_ratio = mask_ratio_to_keep_ratio(mask_ratio)

    for run_id in range(1, num_repeats + 1):
        print("\n" + "=" * 80)
        print(f"Running experiment | mask_ratio={mask_ratio:.2f} | keep_ratio={keep_ratio:.2f} | run={run_id}")
        print("=" * 80)

        # Update config for this run
        CFG.privacy_budget_k = keep_ratio
        CFG.seed = 42 + run_id
        CFG.save_dir = f"{base_save_root}/mask_{int(mask_ratio*100)}/run_{run_id}"

        # Run training + evaluation
        main(CFG)

        # Read saved predictions and compute summary metrics
        pred_path = f"{CFG.save_dir}/test_predictions.csv"
        pred_df = pd.read_csv(pred_path)

        utility_acc = accuracy_score(pred_df["utility_true"], pred_df["utility_pred"])
        utility_f1 = f1_score(pred_df["utility_true"], pred_df["utility_pred"], average="macro")

        all_results.append({
            "mask_ratio": mask_ratio,
            "keep_ratio": keep_ratio,
            "run": run_id,
            "utility_accuracy": utility_acc,
            "utility_f1_macro": utility_f1,
            "save_dir": CFG.save_dir,
        })

results_df = pd.DataFrame(all_results)
results_df


Running experiment | mask_ratio=0.25 | keep_ratio=0.75 | run=1
Using device: cpu
Reading CSV from: /content/drive/My Drive/Collaboration_XAI/XAI_IS/All_CS_rifat_data.csv
CSV shape: (20105, 43)
Columns: ['Convergence_distance', 'fms', 'Left_Eye_Openness', 'Right_Eye_Openness', 'LeftPupilDiameter', 'RightPupilDiameter', 'LeftPupilPosInSensorX', 'LeftPupilPosInSensorY', 'RightPupilPosInSensorX', 'RightPupilPosInSensorY', 'GazeOriginLclSpc_X', 'GazeOriginLclSpc_Y', 'GazeOriginLclSpc_Z', 'GazeDirectionLclSpc_X', 'GazeDirectionLclSpc_Y', 'GazeDirectionLclSpc_Z', 'GazeOriginWrldSpc_X', 'GazeOriginWrldSpc_Y', 'GazeOriginWrldSpc_Z', 'GazeDirectionWrldSpc_X', 'GazeDirectionWrldSpc_Y', 'GazeDirectionWrldSpc_Z', 'NrmLeftEyeOriginX', 'NrmLeftEyeOriginY', 'NrmLeftEyeOriginZ', 'NrmRightEyeOriginX', 'NrmRightEyeOriginY', 'NrmRightEyeOriginZ', 'NrmSRLeftEyeGazeDirX', 'NrmSRLeftEyeGazeDirY', 'NrmSRLeftEyeGazeDirZ', 'NrmSRRightEyeGazeDirX', 'NrmSRRightEyeGazeDirY', 'NrmSRRightEyeGazeDirZ', 'HeadQRotatio

/tmp/ipykernel_16151/3451884794.py:42: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)



========== Stage 1A: Baseline Utility Transformer ==========
[Stage1-Utility][Epoch 01] train_loss=1.3294, train_acc=0.4356, train_f1=0.3184, val_loss=1.2187, val_acc=0.3723, val_f1=0.2784
[Stage1-Utility][Epoch 02] train_loss=1.1829, train_acc=0.5120, train_f1=0.4109, val_loss=0.9535, val_acc=0.5319, val_f1=0.4732
[Stage1-Utility][Epoch 03] train_loss=1.0380, train_acc=0.5906, train_f1=0.4995, val_loss=1.0388, val_acc=0.4255, val_f1=0.3609
[Stage1-Utility][Epoch 04] train_loss=0.9267, train_acc=0.5986, train_f1=0.5198, val_loss=0.9338, val_acc=0.5745, val_f1=0.4970
[Stage1-Utility][Epoch 05] train_loss=0.8501, train_acc=0.6556, train_f1=0.5821, val_loss=0.8058, val_acc=0.5798, val_f1=0.5160
[Stage1-Utility][Epoch 06] train_loss=0.7472, train_acc=0.6705, train_f1=0.6091, val_loss=0.9022, val_acc=0.6170, val_f1=0.5311
[Stage1-Utility][Epoch 07] train_loss=0.7081, train_acc=0.6762, train_f1=0.6251, val_loss=1.0214, val_acc=0.5638, val_f1=0.4824
[Stage1-Utility][Epoch 08] train_loss=0.72

/tmp/ipykernel_16151/3451884794.py:42: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


[Stage2][Epoch 01] main_loss=-0.1579, adv_loss=0.0000, val_util_acc=0.7606, val_util_f1=0.6618, keep_ratio=0.7599
[Stage2][Epoch 02] main_loss=-0.3482, adv_loss=0.0000, val_util_acc=0.7872, val_util_f1=0.6898, keep_ratio=0.7558
[Stage2][Epoch 03] main_loss=-0.3760, adv_loss=0.0000, val_util_acc=0.7713, val_util_f1=0.6711, keep_ratio=0.7516
[Stage2][Epoch 04] main_loss=-0.4086, adv_loss=0.0000, val_util_acc=0.7819, val_util_f1=0.6781, keep_ratio=0.7510
[Stage2][Epoch 05] main_loss=-0.4284, adv_loss=0.0000, val_util_acc=0.7872, val_util_f1=0.6866, keep_ratio=0.7538
[Stage2][Epoch 06] main_loss=-0.4664, adv_loss=0.0000, val_util_acc=0.7872, val_util_f1=0.6863, keep_ratio=0.7478
[Stage2][Epoch 07] main_loss=-0.4758, adv_loss=0.0000, val_util_acc=0.7979, val_util_f1=0.6916, keep_ratio=0.7556
[Stage2][Epoch 08] main_loss=-0.4967, adv_loss=0.0000, val_util_acc=0.8085, val_util_f1=0.6980, keep_ratio=0.7497
[Stage2][Epoch 09] main_loss=-0.5299, adv_loss=0.0000, val_util_acc=0.7979, val_util_f1=

...

[Stage2][Epoch 01] main_loss=1.2439, adv_loss=0.0000, val_util_acc=0.6489, val_util_f1=0.5361, keep_ratio=0.2504
[Stage2][Epoch 02] main_loss=0.4084, adv_loss=0.0000, val_util_acc=0.7074, val_util_f1=0.5661, keep_ratio=0.2294
[Stage2][Epoch 03] main_loss=0.2196, adv_loss=0.0000, val_util_acc=0.6862, val_util_f1=0.5619, keep_ratio=0.2283
[Stage2][Epoch 04] main_loss=0.1917, adv_loss=0.0000, val_util_acc=0.6915, val_util_f1=0.5930, keep_ratio=0.2327
[Stage2][Epoch 05] main_loss=0.1531, adv_loss=0.0000, val_util_acc=0.7181, val_util_f1=0.5998, keep_ratio=0.2427
[Stage2][Epoch 06] main_loss=0.1096, adv_loss=0.0000, val_util_acc=0.7234, val_util_f1=0.5942, keep_ratio=0.2329
[Stage2][Epoch 07] main_loss=0.0796, adv_loss=0.0000, val_util_acc=0.7234, val_util_f1=0.5924, keep_ratio=0.2367
[Stage2][Epoch 08] main_loss=0.0733, adv_loss=0.0000, val_util_acc=0.7287, val_util_f1=0.5943, keep_ratio=0.2384
[Stage2][Epoch 09] main_loss=0.0254, adv_loss=0.0000, val_util_acc=0.7234, val_util_f1=0.6256, k

/tmp/ipykernel_16151/3451884794.py:42: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


[Stage1-Utility][Epoch 01] train_loss=1.3298, train_acc=0.4550, train_f1=0.3531, val_loss=1.1437, val_acc=0.6277, val_f1=0.4563
[Stage1-Utility][Epoch 02] train_loss=1.1197, train_acc=0.5211, train_f1=0.4297, val_loss=1.0510, val_acc=0.6277, val_f1=0.4960
[Stage1-Utility][Epoch 03] train_loss=1.0730, train_acc=0.5735, train_f1=0.4754, val_loss=1.0878, val_acc=0.5426, val_f1=0.4813
[Stage1-Utility][Epoch 04] train_loss=0.9246, train_acc=0.6249, train_f1=0.5315, val_loss=1.0125, val_acc=0.5851, val_f1=0.5217
[Stage1-Utility][Epoch 05] train_loss=0.8078, train_acc=0.6408, train_f1=0.5848, val_loss=0.9851, val_acc=0.5319, val_f1=0.4866
[Stage1-Utility][Epoch 06] train_loss=0.7589, train_acc=0.6522, train_f1=0.5962, val_loss=0.8578, val_acc=0.6223, val_f1=0.5993
[Stage1-Utility][Epoch 07] train_loss=0.6749, train_acc=0.6842, train_f1=0.6405, val_loss=0.9643, val_acc=0.6383, val_f1=0.6116
[Stage1-Utility][Epoch 08] train_loss=0.6026, train_acc=0.7047, train_f1=0.6744, val_loss=1.3037, val_ac

/tmp/ipykernel_16151/3451884794.py:42: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


[Stage2][Epoch 01] main_loss=1.1843, adv_loss=0.0000, val_util_acc=0.5904, val_util_f1=0.5115, keep_ratio=0.2838
[Stage2][Epoch 02] main_loss=0.4611, adv_loss=0.0000, val_util_acc=0.6170, val_util_f1=0.5602, keep_ratio=0.2354
[Stage2][Epoch 03] main_loss=0.3398, adv_loss=0.0000, val_util_acc=0.6702, val_util_f1=0.6242, keep_ratio=0.2298
[Stage2][Epoch 04] main_loss=0.2620, adv_loss=0.0000, val_util_acc=0.6117, val_util_f1=0.5546, keep_ratio=0.2278
[Stage2][Epoch 05] main_loss=0.1923, adv_loss=0.0000, val_util_acc=0.6702, val_util_f1=0.6165, keep_ratio=0.2342
[Stage2][Epoch 06] main_loss=0.2022, adv_loss=0.0000, val_util_acc=0.6543, val_util_f1=0.5157, keep_ratio=0.2342
[Stage2][Epoch 07] main_loss=0.1791, adv_loss=0.0000, val_util_acc=0.6968, val_util_f1=0.6419, keep_ratio=0.2378
[Stage2][Epoch 08] main_loss=0.1559, adv_loss=0.0000, val_util_acc=0.6649, val_util_f1=0.5783, keep_ratio=0.2185
[Stage2][Epoch 09] main_loss=0.1367, adv_loss=0.0000, val_util_acc=0.6596, val_util_f1=0.5843, k

In [ ]:
# ============================================================
# Cell 9: Save all results
# ============================================================
results_df.to_csv("/content/cybersickness_masking_experiments/all_masking_results.csv", index=False)
print(results_df)

print("\nSaved summary to:")
print("/content/cybersickness_masking_experiments/all_masking_results.csv")

In [ ]:
# ============================================================
# Cell 10: Mean result for each masking ratio
# ============================================================
summary_df = (
    results_df.groupby(["mask_ratio", "keep_ratio"], as_index=False)
    .agg(
        utility_accuracy_mean=("utility_accuracy", "mean"),
        utility_accuracy_std=("utility_accuracy", "std"),
        utility_f1_macro_mean=("utility_f1_macro", "mean"),
        utility_f1_macro_std=("utility_f1_macro", "std"),
    )
)

summary_df

In [ ]:
# ============================================================
# Cell 11: Optional - sort nicely
# ============================================================
summary_df = summary_df.sort_values("mask_ratio").reset_index(drop=True)
print(summary_df)